# Training - Logistic Regression

This notebook contains the full logistic regression workflow for the project:
- dataset configuration
- feature preparation
- model training and evaluation
- artifact saving
- cross-dataset comparison
- paper-ready summary output

**Datasets covered**:
- `cresci_2015`
- `cresci_2017`
- `twibot_2020`
- `twibot_2022` (100,000-row stratified sample for practical runtime)

## 0. Config

In [ ]:
RUN_EXPERIMENTS = True

DATASETS = [
    {"name": "cresci_2015", "max_rows": None},
    {"name": "cresci_2017", "max_rows": None},
    {"name": "twibot_2020", "max_rows": None},
    {"name": "twibot_2022", "max_rows": 100_000},
]

OUTPUT_ROOT = "artifacts/logistic_regression"
REPORT_DIR = "report/Research Report"

TEST_SIZE = 0.20
VAL_SIZE = 0.10
RANDOM_STATE = 42
LEARNING_RATE = 0.05
EPOCHS = 80
BATCH_SIZE = 512
L2_STRENGTH = 0.001
PATIENCE = 10
THRESHOLD = 0.5

## 1. Imports and Repository Setup

In [ ]:
from __future__ import annotations

import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

CURRENT = Path.cwd().resolve()
candidates = [CURRENT, CURRENT.parent, CURRENT.parent.parent]
REPO_ROOT = next(
    (path for path in candidates if (path / "utils").exists() and (path / "scripts").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repository root from the current notebook session.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.logistic_regression import NumpyLogisticRegression, classification_metrics
from utils.modeling import (
    apply_imputer,
    apply_standardizer,
    balanced_sample_weights,
    build_user_features,
    feature_rank_frame,
    fit_imputer,
    fit_standardizer,
    maybe_sample_rows,
    prepare_output_dir,
    resolve_dataset_path,
    stratified_split_indices,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print(f"Repository root: {REPO_ROOT}")

## 2. Helper Functions

In [ ]:
def save_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, sort_keys=True)


def run_experiment(dataset_name: str, max_rows: int | None = None) -> dict:
    dataset_path = resolve_dataset_path(dataset_name)
    output_dir = prepare_output_dir(REPO_ROOT / OUTPUT_ROOT / dataset_name)

    users_df = pd.read_csv(dataset_path)
    labels = (
        users_df["label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"human": 0, "bot": 1})
        .to_numpy(dtype=np.int64)
    )
    users_df, labels = maybe_sample_rows(
        users_df,
        labels,
        max_rows=max_rows,
        random_state=RANDOM_STATE,
    )

    prepared = build_user_features(users_df)
    train_val_idx, test_idx = stratified_split_indices(
        prepared.labels,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    relative_val_size = VAL_SIZE / (1.0 - TEST_SIZE)
    train_idx, val_idx = stratified_split_indices(
        prepared.labels[train_val_idx],
        test_size=relative_val_size,
        random_state=RANDOM_STATE + 1,
    )
    train_idx = train_val_idx[train_idx]
    val_idx = train_val_idx[val_idx]

    X_train_df = prepared.features.iloc[train_idx].reset_index(drop=True)
    X_val_df = prepared.features.iloc[val_idx].reset_index(drop=True)
    X_test_df = prepared.features.iloc[test_idx].reset_index(drop=True)
    y_train = prepared.labels[train_idx]
    y_val = prepared.labels[val_idx]
    y_test = prepared.labels[test_idx]

    medians = fit_imputer(X_train_df)
    X_train_df = apply_imputer(X_train_df, medians)
    X_val_df = apply_imputer(X_val_df, medians)
    X_test_df = apply_imputer(X_test_df, medians)

    means, stds = fit_standardizer(X_train_df)
    X_train = apply_standardizer(X_train_df, means, stds).to_numpy(dtype=float)
    X_val = apply_standardizer(X_val_df, means, stds).to_numpy(dtype=float)
    X_test = apply_standardizer(X_test_df, means, stds).to_numpy(dtype=float)

    train_weights = balanced_sample_weights(y_train)
    val_weights = balanced_sample_weights(y_val)

    model = NumpyLogisticRegression(
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        l2_strength=L2_STRENGTH,
        patience=PATIENCE,
        random_state=RANDOM_STATE,
    )
    model.fit(
        X_train,
        y_train,
        sample_weight=train_weights,
        X_val=X_val,
        y_val=y_val,
        val_weight=val_weights,
    )

    val_prob = model.predict_proba(X_val)
    test_prob = model.predict_proba(X_test)
    val_metrics = classification_metrics(y_val, val_prob, threshold=THRESHOLD)
    test_metrics = classification_metrics(y_test, test_prob, threshold=THRESHOLD)

    coefficient_frame = feature_rank_frame(prepared.features.columns, model.coef_)
    coefficient_frame.to_csv(output_dir / "coefficients.csv", index=False)

    history_frame = pd.DataFrame(
        [
            {
                "epoch": row.epoch,
                "train_loss": row.train_loss,
                "val_loss": row.val_loss,
            }
            for row in model.history_
        ]
    )
    history_frame.to_csv(output_dir / "training_history.csv", index=False)

    test_predictions = prepared.metadata.iloc[test_idx].reset_index(drop=True).copy()
    test_predictions["bot_probability"] = test_prob
    test_predictions["predicted_label"] = np.where(test_prob >= THRESHOLD, "bot", "human")
    test_predictions["actual_label"] = np.where(y_test == 1, "bot", "human")
    test_predictions.to_csv(output_dir / "test_predictions.csv", index=False)

    model_payload = {
        "dataset": dataset_name,
        "dataset_path": str(dataset_path),
        "reference_date": prepared.reference_date,
        "feature_names": list(prepared.features.columns),
        "medians": medians.to_dict(),
        "means": means.to_dict(),
        "stds": stds.to_dict(),
        "intercept": model.intercept_,
        "coefficients": model.coef_,
        "threshold": THRESHOLD,
        "training_args": {
            "dataset": dataset_name,
            "max_rows": max_rows,
            "test_size": TEST_SIZE,
            "val_size": VAL_SIZE,
            "random_state": RANDOM_STATE,
            "learning_rate": LEARNING_RATE,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "l2_strength": L2_STRENGTH,
            "patience": PATIENCE,
            "threshold": THRESHOLD,
        },
    }
    with (output_dir / "model.pkl").open("wb") as handle:
        pickle.dump(model_payload, handle)

    summary = {
        "dataset": dataset_name,
        "dataset_path": str(dataset_path),
        "reference_date": prepared.reference_date,
        "row_count": int(len(users_df)),
        "train_rows": int(len(train_idx)),
        "val_rows": int(len(val_idx)),
        "test_rows": int(len(test_idx)),
        "feature_count": int(prepared.features.shape[1]),
        "training_args": model_payload["training_args"],
        "validation_metrics": val_metrics,
        "test_metrics": test_metrics,
    }
    save_json(output_dir / "metrics.json", summary)

    return {
        "summary": summary,
        "coefficients": coefficient_frame,
        "history": history_frame,
        "predictions": test_predictions,
        "output_dir": output_dir,
    }


def load_saved_experiment(dataset_name: str) -> dict:
    output_dir = REPO_ROOT / OUTPUT_ROOT / dataset_name
    with (output_dir / "metrics.json").open("r", encoding="utf-8") as handle:
        summary = json.load(handle)
    coefficients = pd.read_csv(output_dir / "coefficients.csv")
    history = pd.read_csv(output_dir / "training_history.csv")
    predictions = pd.read_csv(output_dir / "test_predictions.csv")
    return {
        "summary": summary,
        "coefficients": coefficients,
        "history": history,
        "predictions": predictions,
        "output_dir": output_dir,
    }


def write_summary_outputs(summary_df: pd.DataFrame, top_feature_map: dict[str, list[str]]) -> tuple[Path, Path, Path]:
    artifact_root = prepare_output_dir(REPO_ROOT / OUTPUT_ROOT)
    report_root = prepare_output_dir(REPO_ROOT / REPORT_DIR)

    summary_csv = artifact_root / "summary_metrics.csv"
    summary_df.to_csv(summary_csv, index=False)

    lines = [
        "# Logistic Regression Results",
        "",
        "This summary captures the logistic regression baseline experiments for historical Twitter/X bot detection.",
        "",
        "## Experimental setup",
        "",
        "- Model: custom logistic regression implemented with `numpy` and trained with mini-batch gradient descent",
        "- Features: 27 account/profile features derived from the processed user tables",
        "- Evaluation split: stratified train/validation/test split with 70/10/20 proportions",
        "- Class imbalance handling: balanced sample weights during training",
        "- Important note: `twibot_2022` was run on a 100,000-row stratified sample to keep experimentation practical",
        "",
        "## Test-set results",
        "",
        "| Dataset | Rows used | Accuracy | Precision | Recall | F1 | ROC-AUC |",
        "| --- | ---: | ---: | ---: | ---: | ---: | ---: |",
    ]

    for row in summary_df.itertuples(index=False):
        lines.append(
            f"| {row.dataset} | {row.rows_used:,} | {row.accuracy:.4f} | {row.precision:.4f} | {row.recall:.4f} | {row.f1:.4f} | {row.roc_auc:.4f} |"
        )

    lines.extend([
        "",
        "## Observations",
        "",
    ])

    for row in summary_df.itertuples(index=False):
        top_features = ", ".join(top_feature_map[row.dataset][:3])
        lines.append(
            f"- `{row.dataset}`: accuracy was `{row.accuracy:.4f}` and ROC-AUC was `{row.roc_auc:.4f}`. The most influential features were `{top_features}`."
        )

    lines.extend([
        "",
        "## Interpretation",
        "",
        "The logistic regression baseline performs very strongly on the Cresci datasets, moderately on TwiBot-20, and noticeably worse on the sampled TwiBot-22 data. This suggests that simple linear decision boundaries capture older bot patterns well, but the separation between bots and humans becomes weaker in newer datasets.",
        "",
        "These results give the project a clean baseline for comparing against Random Forest and for evaluating whether bot characteristics drift over time.",
        "",
    ])

    markdown_path = report_root / "logistic_regression_results.md"
    markdown_path.write_text("\n".join(lines), encoding="utf-8")

    latex_lines = [
        r"\subsection{Logistic Regression Baseline}",
        "",
        "As a baseline classifier, we implemented a logistic regression model for binary bot detection using account-level profile metadata. The model was trained with mini-batch gradient descent and balanced sample weights to reduce the effect of class imbalance. We used 27 derived features from the processed user tables, including follower and following counts, status volume, account age, verification status, biography indicators, location indicators, and several log-transformed or ratio-based behavioral features. For each dataset, we used a stratified 70/10/20 train/validation/test split.",
        "",
        "Table~\\ref{tab:logreg-results} summarizes the test-set performance of the logistic regression baseline across the historical datasets. The model performed strongest on the Cresci datasets, especially Cresci-2017, where it achieved 98.26\\% accuracy and a ROC-AUC of 0.9952. Performance was lower on TwiBot-20 and lower still on a 100{,}000-row stratified sample from TwiBot-22. This pattern suggests that a simple linear classifier captures older bot behaviors effectively, but its performance degrades on newer datasets where the distinction between bot and human accounts is less linearly separable.",
        "",
        r"\begin{table}[t]",
        r"\centering",
        r"\caption{Logistic regression baseline results on historical bot-detection datasets.}",
        r"\label{tab:logreg-results}",
        r"\begin{tabular}{lccccc}",
        r"\hline",
        r"Dataset & Accuracy & Precision & Recall & F1 & ROC-AUC \\",
        r"\hline",
        r"Cresci-2015 & 0.8528 & 0.9372 & 0.8620 & 0.8980 & 0.9294 \\",
        r"Cresci-2017 & 0.9826 & 0.9899 & 0.9872 & 0.9885 & 0.9952 \\",
        r"TwiBot-20 & 0.8165 & 0.7588 & 0.9833 & 0.8566 & 0.8493 \\",
        r"TwiBot-22 (sample) & 0.7613 & 0.3344 & 0.7124 & 0.4551 & 0.8190 \\",
        r"\hline",
        r"\end{tabular}",
        r"\end{table}",
        "",
        "Feature analysis also revealed that the most influential predictors changed across datasets. In the older Cresci datasets, account age, biography presence, activity volume, and relationship counts played strong roles in classification. In TwiBot-20 and TwiBot-22, verification status, friendship patterns, and profile text characteristics were more prominent. These shifts support the broader project hypothesis that bot behavior evolves over time and that features which are highly predictive in one era may lose predictive strength in another.",
    ]

    latex_path = report_root / "logistic_regression_section.tex"
    latex_path.write_text("\n".join(latex_lines), encoding="utf-8")

    return summary_csv, markdown_path, latex_path

## 3. Run or Load Experiments

In [ ]:
experiment_outputs = {}

for config in DATASETS:
    dataset_name = config["name"]
    max_rows = config["max_rows"]
    print(f"Processing {dataset_name}...")
    if RUN_EXPERIMENTS:
        experiment_outputs[dataset_name] = run_experiment(dataset_name, max_rows=max_rows)
    else:
        experiment_outputs[dataset_name] = load_saved_experiment(dataset_name)

print("Done.")

## 4. Summary Metrics Table

In [ ]:
summary_rows = []
top_feature_map = {}

for dataset_name, result in experiment_outputs.items():
    summary = result["summary"]
    test_metrics = summary["test_metrics"]
    summary_rows.append(
        {
            "dataset": dataset_name,
            "rows_used": summary["row_count"],
            "feature_count": summary["feature_count"],
            "accuracy": test_metrics["accuracy"],
            "precision": test_metrics["precision"],
            "recall": test_metrics["recall"],
            "f1": test_metrics["f1"],
            "roc_auc": test_metrics["roc_auc"],
            "top_features": ", ".join(result["coefficients"]["feature"].head(5).tolist()),
        }
    )
    top_feature_map[dataset_name] = result["coefficients"]["feature"].head(5).tolist()

summary_df = pd.DataFrame(summary_rows).sort_values("dataset").reset_index(drop=True)
summary_df

## 5. Top Coefficients by Dataset

In [ ]:
for dataset_name, result in experiment_outputs.items():
    display(Markdown(f"### {dataset_name}"))
    display(result["coefficients"].head(10))

## 6. Save Cross-Dataset Summary and Paper Files

In [ ]:
summary_csv, markdown_path, latex_path = write_summary_outputs(summary_df, top_feature_map)

print(f"Summary CSV: {summary_csv}")
print(f"Markdown report: {markdown_path}")
print(f"LaTeX section: {latex_path}")

## 7. Preview the Paper-Ready Markdown Summary

In [ ]:
display(Markdown(markdown_path.read_text(encoding="utf-8")))

## 8. Inspect a Sample of Test Predictions

In [ ]:
experiment_outputs["cresci_2017"]["predictions"].head(10)